In [5]:
# ============================================================
# TREINAMENTO DO MODELO DE LIMITE DE CRÉDITO
# ============================================================

import pandas as pd
import tensorflow as tf
from tensorflow import keras

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

import numpy as np
import joblib


# ============================================================
# 1. Carregar a base de dados
# ============================================================

df = pd.read_csv('limites_de_credito_1000_registros.csv')

# Variáveis de entrada
# São os dados que o modelo receberá para calcular o limite
X = df[['Idade', 'Salario', 'Score_Credito']]

# Variável alvo
# É o valor que o modelo deve aprender a prever
y = df['Limite_Aprovado']


# ============================================================
# 2. Separar treino e teste
# ============================================================

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


# ============================================================
# 3. Normalizar os dados
# ============================================================

# Normalizador das entradas
normalizador_x = StandardScaler()

X_treino_norm = normalizador_x.fit_transform(X_treino)
X_teste_norm = normalizador_x.transform(X_teste)


# Normalizador do alvo
normalizador_y = StandardScaler()

y_treino_norm = normalizador_y.fit_transform(y_treino.values.reshape(-1, 1))
y_teste_norm = normalizador_y.transform(y_teste.values.reshape(-1, 1))


# ============================================================
# 4. Construir a rede neural
# ============================================================

tf.random.set_seed(42)

modelo_limite = keras.Sequential([
    keras.Input(shape=(3,)),

    keras.layers.Dense(16, activation='relu'),
    keras.layers.Dense(8, activation='relu'),

    # Saída com 1 neurônio, pois queremos prever um valor numérico
    keras.layers.Dense(1)
])

modelo_limite.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)


# ============================================================
# 5. Treinar o modelo
# ============================================================

print("Treinando o modelo de limite de crédito...\n")

historico = modelo_limite.fit(
    X_treino_norm,
    y_treino_norm,
    epochs=100,
    validation_split=0.2,
    verbose=0
)

print("\nTreinamento concluído!")


# ============================================================
# 6. Avaliar o modelo
# ============================================================

perda, mae = modelo_limite.evaluate(X_teste_norm, y_teste_norm)

print(f"\nPerda no teste em escala normalizada: {perda:.4f}")
print(f"MAE no teste em escala normalizada: {mae:.4f}")


# Fazendo previsões no conjunto de teste
previsoes_norm = modelo_limite.predict(X_teste_norm)

# Convertendo previsões para Reais
previsoes_reais = normalizador_y.inverse_transform(previsoes_norm)

# Valores reais do teste
y_teste_reais = y_teste.values.reshape(-1, 1)

# Calculando erro em Reais
mae_reais = mean_absolute_error(y_teste_reais, previsoes_reais)
rmse_reais = np.sqrt(mean_squared_error(y_teste_reais, previsoes_reais))

print(f"Erro médio absoluto em Reais: R$ {mae_reais:.2f}")
print(f"RMSE em Reais: R$ {rmse_reais:.2f}")


# ============================================================
# 7. Salvar o modelo e os normalizadores
# ============================================================

# Salva o modelo treinado
modelo_limite.save('modelo_limite_credito.keras')

# Salva o normalizador das entradas
joblib.dump(normalizador_x, 'normalizador_x.pkl')

# Salva o normalizador do alvo
joblib.dump(normalizador_y, 'normalizador_y.pkl')

print("\nArquivos salvos com sucesso:")
print("- modelo_limite_credito.keras")
print("- normalizador_x.pkl")
print("- normalizador_y.pkl")

Treinando o modelo de limite de crédito...


Treinamento concluído!
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0763 - mae: 0.2235  

Perda no teste em escala normalizada: 0.0763
MAE no teste em escala normalizada: 0.2235
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
Erro médio absoluto em Reais: R$ 604.15
RMSE em Reais: R$ 746.85

Arquivos salvos com sucesso:
- modelo_limite_credito.keras
- normalizador_x.pkl
- normalizador_y.pkl


In [6]:
!uname -a

Linux 54754488398a 6.6.122+ #1 SMP Thu Apr 30 18:17:14 UTC 2026 x86_64 x86_64 x86_64 GNU/Linux
